# Image Classifier — MobileNetV2
**Cloud Computing Project — AWS Lambda**

In [ ]:
!pip install torch torchvision pillow requests --quiet
print('✅ Libraries installed')

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from PIL import Image
import requests, json
from io import BytesIO

print('Loading MobileNetV2...')
weights = MobileNet_V2_Weights.IMAGENET1K_V1
model = mobilenet_v2(weights=weights)
model.eval()
categories = weights.meta['categories']
print(f'✅ Model loaded — {len(categories)} classes')

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def classify_image(img: Image.Image):
    img = img.convert('RGB')
    input_tensor = preprocess(img).unsqueeze(0)
    with torch.no_grad():
        output = model(input_tensor)
    probs = torch.nn.functional.softmax(output[0], dim=0)
    top5_prob, top5_idx = torch.topk(probs, 5)
    return [
        {'label': categories[idx], 'confidence': round(p.item() * 100, 2)}
        for p, idx in zip(top5_prob, top5_idx)
    ]

print('✅ classify_image() ready')

In [ ]:
from google.colab import files
from IPython.display import display

uploaded = files.upload()

for filename in uploaded:
    img = Image.open(BytesIO(uploaded[filename]))
    display(img)
    results = classify_image(img)
    print(f'\n🔍 Results for {filename}:')
    for i, r in enumerate(results, 1):
        bar = '█' * int(r['confidence'] / 4)
        print(f"  {i}. {r['label']:30s} {r['confidence']:5.1f}%  {bar}")

In [ ]:
def lambda_handler(event, context=None):
    try:
        image_url = event.get('image_url')
        if not image_url:
            return {'statusCode': 400, 'body': json.dumps({'error': 'image_url is required'})}

        headers = {'User-Agent': 'Mozilla/5.0'}
        resp = requests.get(image_url, headers=headers, timeout=8)
        img = Image.open(BytesIO(resp.content))
        predictions = classify_image(img)

        return {
            'statusCode': 200,
            'body': json.dumps({'predictions': predictions, 'model': 'MobileNetV2-ImageNet1K'})
        }
    except Exception as e:
        return {'statusCode': 500, 'body': json.dumps({'error': str(e)})}


event = {'image_url': 'https://images.dog.ceo/breeds/retriever-golden/n02099601_3004.jpg'}
response = lambda_handler(event)
print('Status:', response['statusCode'])
body = json.loads(response['body'])
print('Top prediction:', body['predictions'][0])